In [1]:
%pip install -U accelerate 
%pip install -U peft 
%pip install -U trl 
%pip install -U bitsandbytes
%pip install -U transformers


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from huggingface_hub import login
import os
#REDACTED_OPENAI_KEY
hf_token = os.environ.get("HF_TOKEN")
login(hf_token)

In [3]:

import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# -----------------------------
# 0) (권장) 환경 변수
# -----------------------------
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# MPS에서 일부 연산 fallback 허용(에러 줄이는 용도)
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# -----------------------------
# 1) 시스템 프롬프트
# -----------------------------
SYSTEM_PROMPT = PROMPT = """\
다양한 작업에 대한 답변을 생성해주세요. 이러한 작업 지침은 ChatGPT 모델에 주어지며, ChatGPT 모델이 지침을 완료하는지 평가합니다.

요구 사항은 다음과 같습니다:
1. 다양성을 극대화하기 위해 각 지시에 대해 동사를 반복하지 않도록 하세요.
2. 지시에 사용되는 언어도 다양해야 합니다. 예를 들어, 질문과 명령형 지시를 결합해야 합니다.
3. 지시 사항의 유형이 다양해야 합니다. 목록에는 개방형 생성, 분류, 편집 등과 같은 다양한 유형의 작업이 포함되어야 합니다.
2. GPT 언어 모델은 지시를 완료할 수 있어야 합니다. 예를 들어 어시스턴트에게 시각적 또는 오디오 출력을 생성하도록 요청하지 마세요. 또 다른 예로, 어시스턴트가 어떤 작업도 수행할 수 없으므로 오후 5시에 깨우거나 미리 알림을 설정하도록 요청하지 마세요.
3. 답변은 한국어로 작성해야 합니다.
4. 답변을 1~2문장으로 작성하세요. 명령문이나 질문도 허용됩니다.
5. 지시 사항에 대한 적절한 입력을 생성해야 합니다. 입력 필드에는 지시에 대한 구체적인 예가 포함되어야 합니다. 실제 데이터를 포함해야 하며 단순한 자리 표시자를 포함해서는 안 됩니다. 입력은 지시 사항을 어렵게 만들 수 있는 상당한 내용을 제공해야 하지만 100단어를 넘지 않는 것이 이상적입니다.
6. 일부 지시사항은 추가 입력이 있고, 일부 지시에는 입력 필드가 비어있습니다. 예를 들어 "세계에서 가장 높은 봉우리는 무엇인가?"라는 일반적인 정보를 묻는 지시의 경우 구체적인 맥락을 제공할 필요가 없어, 입력 필드가 비어있을 수 있습니다.
7. 출력은 명령어와 입력에 대한 적절한 응답이어야 합니다.

아래에 10개의 명령어와 입력(옵션)에 따라 적절한 응답을 생성하세요. 
응답은 아래와 같은 형식으로 10가지를 0번 부터 9번 까지, 번호에 따라 해당 번호의 명령어와 입력에 알맞게 작성하세요.

각 응답 사이는 ### 으로 내용을 분리해주세요.

응답0: 첫 번째 응답내용###
응답1: 두 번째 응답내용###
...
응답9: 마지막 응답내용""".strip()

# -----------------------------
# 2) HF Dataset 로딩 (KoAlpaca)
# -----------------------------
def load_hf_dataset(repo_id: str = "beomi/KoAlpaca-v1.1a", split: str = "train"):
    ds = load_dataset(repo_id, split=split)
    return ds


# -----------------------------
# 3) Dataset -> SFT용 text 컬럼 생성
#    (기존 방식 유지: apply_chat_template로 "text" 만들기)
# -----------------------------
def build_sft_text_dataset(hf_ds, tokenizer, system_prompt: str):
    def _format_one(example):
        instruction = (example.get("instruction") or "").strip()
        user_input = (example.get("input") or "").strip()
        output = (example.get("output") or "").strip()

        # KoAlpaca는 instruction + (optional) input 구조
        if user_input:
            user_content = f"{instruction}\n\n[추가 입력]\n{user_input}"
        else:
            user_content = instruction

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        return {"text": text}

    # 원본 컬럼들 제거하고 text만 남기기
    text_ds = hf_ds.map(
        _format_one,
        remove_columns=hf_ds.column_names,
        desc="Formatting dataset to SFT text",
    )
    return text_ds

# -----------------------------
# 4) 모델 & 토크나이저 로딩 (MPS + Qwen2.5 + LoRA)
# -----------------------------
def load_model_and_tokenizer(model_id: str = "Qwen/Qwen2.5-1.5B-Instruct"):
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    device = "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"Using device: {device}")

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float32,  # MPS 안정적으로 fp32
    )

    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, lora_config)
    model.to(device)
    model.print_trainable_parameters()

    return model, tokenizer

# -----------------------------
# 5) SFTTrainer 학습
# -----------------------------
def train():
    # (1) 데이터셋 로드
    hf_ds = load_hf_dataset(
        "beomi/KoAlpaca-v1.1a",
        split="train[:1000]")
    
    # (2) 모델/토크나이저 로드
    model, tokenizer = load_model_and_tokenizer("Qwen/Qwen2.5-1.5B-Instruct")

    # (3) SFT용 text dataset 만들기
    train_ds = build_sft_text_dataset(hf_ds, tokenizer, SYSTEM_PROMPT)

    # (4) 학습 설정
    args = SFTConfig(
        output_dir="qwen2_5_1_5b_koalpaca_lora_mps_sft_1",
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_ratio=0.1,
        learning_rate=1e-5,

        optim="adamw_torch",  # MPS에서 fused보다 안정적인 편
        fp16=False,
        bf16=False,

        logging_steps=1,
        save_strategy="epoch",
        report_to="none",

        # 길이/효율
        max_length=1024,   # TRL 최신: max_seq_length 대신 max_length
        packing=True,      # 짧은 샘플 많으면 효율 향상

        gradient_checkpointing=True,
        remove_unused_columns=False,  # 안전하게
        shuffle_dataset=True,
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        processing_class=tokenizer,  # 최신 TRL에서 tokenizer 대신 processing_class
    )

    trainer.train()

    # (5) LoRA adapter 저장
    save_dir = os.path.join(args.output_dir, "adapter")
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print("저장 완료:", save_dir)

if __name__ == "__main__":
    train()


Using device: mps


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


Formatting dataset to SFT text:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Packing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/opt/homebrew/Caskroom/miniforge/base/envs/llm/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,2.236446
2,2.225324
3,2.201270
4,2.247432
5,2.229380
6,2.180768
7,2.260348
8,2.235938
9,2.222918
10,2.253727


/opt/homebrew/Caskroom/miniforge/base/envs/llm/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/homebrew/Caskroom/miniforge/base/envs/llm/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


저장 완료: qwen2_5_1_5b_koalpaca_lora_mps_sft_1/adapter


In [9]:
import torch
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
import os


# 학습 때 저장한 어댑터 경로
adapter_path = "qwen2_5_1_5b_koalpaca_lora_mps_sft_1/adapter"

# 1. 토크나이저는 어댑터 디렉토리에서 로드 (거기에 base_model 정보도 들어있음)
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# 2. LoRA가 합쳐진 모델 로드 (base + adapter)
model = AutoPeftModelForCausalLM.from_pretrained(
    adapter_path,
    dtype=torch.float16,
    device_map="auto",           # 0.5B라 auto로 올려도 부담 없음
)
model.eval()

print("파인튜닝 모델 로드 완료")


os.environ["GOOGLE_API_KEY"] = "REDACTED_GOOGLE_API_KEY"
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 아까 만든 DB 폴더가 있다면 불러오기만 하면 됩니다.
vectorstore = Chroma(persist_directory="./final_legal_db", embedding_function=embeddings)


# -----------------------------
# 시스템 프롬프트 (학습 때 쓴 것 그대로)
# -----------------------------
SYSTEM_PROMPT = PROMPT = """\
다양한 작업에 대한 답변을 생성해주세요. 이러한 작업 지침은 ChatGPT 모델에 주어지며, ChatGPT 모델이 지침을 완료하는지 평가합니다.

요구 사항은 다음과 같습니다:
1. 다양성을 극대화하기 위해 각 지시에 대해 동사를 반복하지 않도록 하세요.
2. 지시에 사용되는 언어도 다양해야 합니다. 예를 들어, 질문과 명령형 지시를 결합해야 합니다.
3. 지시 사항의 유형이 다양해야 합니다. 목록에는 개방형 생성, 분류, 편집 등과 같은 다양한 유형의 작업이 포함되어야 합니다.
2. GPT 언어 모델은 지시를 완료할 수 있어야 합니다. 예를 들어 어시스턴트에게 시각적 또는 오디오 출력을 생성하도록 요청하지 마세요. 또 다른 예로, 어시스턴트가 어떤 작업도 수행할 수 없으므로 오후 5시에 깨우거나 미리 알림을 설정하도록 요청하지 마세요.
3. 답변은 한국어로 작성해야 합니다.
4. 답변을 1~2문장으로 작성하세요. 명령문이나 질문도 허용됩니다.
5. 지시 사항에 대한 적절한 입력을 생성해야 합니다. 입력 필드에는 지시에 대한 구체적인 예가 포함되어야 합니다. 실제 데이터를 포함해야 하며 단순한 자리 표시자를 포함해서는 안 됩니다. 입력은 지시 사항을 어렵게 만들 수 있는 상당한 내용을 제공해야 하지만 100단어를 넘지 않는 것이 이상적입니다.
6. 일부 지시사항은 추가 입력이 있고, 일부 지시에는 입력 필드가 비어있습니다. 예를 들어 "세계에서 가장 높은 봉우리는 무엇인가?"라는 일반적인 정보를 묻는 지시의 경우 구체적인 맥락을 제공할 필요가 없어, 입력 필드가 비어있을 수 있습니다.
7. 출력은 명령어와 입력에 대한 적절한 응답이어야 합니다.
8. 당신은 법률 문제 해결 도우미이기 때문에 갖고 있는 지식 내에서 사용자에게 명확한 답변을 내려주세요.

아래에 10개의 명령어와 입력(옵션)에 따라 적절한 응답을 생성하세요. 
응답은 아래와 같은 형식으로 10가지를 0번 부터 9번 까지, 번호에 따라 해당 번호의 명령어와 입력에 알맞게 작성하세요.

각 응답 사이는 ### 으로 내용을 분리해주세요.

응답0: 첫 번째 응답내용###
응답1: 두 번째 응답내용###
...
응답9: 마지막 응답내용""".strip()


def chat_with_model_rag(question: str):
    # (A) RAG: 질문과 관련된 법률 지식 검색
    print(f"\n[지식 검색 중...]")
    docs = vectorstore.similarity_search(question, k=3)
    context = "\n\n".join([doc.page_content for doc in docs])

    # (B) 메시지 구성: 검색된 context를 시스템 프롬프트나 사용자 질문에 삽입
    # 사용자님이 설정하신 SYSTEM_PROMPT를 그대로 활용하되 context만 추가합니다.
    combined_user_content = f"다음 법률 문맥을 참고하여 답변하세요:\n\n[법률 문맥]\n{context}\n\n[질문]\n{question}"

    messages = [
        {"role": "system", "content": "당신은 법률 전문 어시스턴트입니다. 제공된 문맥만을 바탕으로 정확하게 답변하세요."},
        {"role": "user", "content": combined_user_content},
    ]

    # Qwen chat template 적용 (사용자님 코드 방식)
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3, # 법률 답변이므로 낮은 온도로 정확성 유지
            do_sample=True,
        )

    gen_ids = gen[0, input_len:]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True)
    print("\n=== 법률 특화 파인튜닝 모델 응답 ===")
    print(answer.strip())

# 실행
if __name__ == "__main__":
    chat_with_model_rag("신호위반 차량이 저를 들이받고 도주했습니다. 가해자가 무보험이라는데 어떻게 되나요?")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

파인튜닝 모델 로드 완료

[지식 검색 중...]

=== 법률 특화 파인튜닝 모델 응답 ===
죄송합니다, 저는 실시간 정보 검색이나 실제 사건에 대한 조사가 불가능한 점에 주의하시기 바랍니다. 그러나 일반적으로 신호위반 차량이 도주하는 경우 보험을 가진 운전자의 책임을 부담할 수 있습니다. 하지만 무보험차량이라면 보험을 가진 운전자와 차량이 분명해야 합니다. 이 경우, 무보험차량의 운전자에게는 보상책임이 없으므로, 그 차량과 운전자에게도 책임이 있음을 확인해야 합니다. 또한, 경찰에 신고하고 사고 현장에서 발견한 모든 자료들을 기록하여 법원에 제출하면 됩니다. 이러한 절차를 통해 피해자로서의 권리를 최대한 보호할 수 있을 것입니다.
